In [1]:
import os
import gc
import math
import random
import unicodedata
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [2]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()

# User paths
LST20_ROOT = "/content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/Corpus-LST20"
COMP_ROOT  = "/content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/super-ai-engineer-ss-6-word-segmentation"
OUT_DIR    = "/content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/submission"

# Training
BATCH_SIZE = 256 if torch.cuda.is_available() else 64
EMBED_DIM = 256
TYPE_EMBED_DIM = 16
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.20
LR = 2e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 8               # tuning stage
PATIENCE = 2
GRAD_CLIP = 1.0
NUM_WORKERS = 2

# Final retrain on train+eval using best epoch from stage-1
FINAL_RETRAIN = True
FINAL_LR = 1.5e-3

# Submission mapping uncertainty:
# We export both variants, and set submission.csv = single-char -> B_WORD by default.
DEFAULT_SINGLE_CHAR_LABEL = "B_WORD"   # can be switched to "E_WORD"

# Threshold grid for end-of-word probability
THRESHOLDS = np.arange(0.35, 0.66, 0.02)


In [3]:
# 2) REPRODUCIBILITY

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
print("DEVICE:", DEVICE)



DEVICE: cuda


In [4]:
# 3) PATH HELPERS

def resolve_lst20_root(root: str) -> Path:
    p = Path(root)
    if (p / "LST20_Corpus").exists():
        p = p / "LST20_Corpus"
    assert p.exists(), f"LST20 root not found: {p}"
    assert (p / "train").exists(), f"train split not found under: {p}"
    return p


def resolve_comp_root(root: str) -> Path:
    p = Path(root)
    assert p.exists(), f"Competition root not found: {p}"
    needed = ["ws_test.txt", "ws_sample_submission.csv", "ws_list.txt"]
    for f in needed:
        assert (p / f).exists(), f"Missing {f} in {p}"
    return p


LST20_ROOT_PATH = resolve_lst20_root(LST20_ROOT)
COMP_ROOT_PATH = resolve_comp_root(COMP_ROOT)
print("LST20_ROOT_PATH:", LST20_ROOT_PATH)
print("COMP_ROOT_PATH:", COMP_ROOT_PATH)


LST20_ROOT_PATH: /content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/Corpus-LST20/LST20_Corpus
COMP_ROOT_PATH: /content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/super-ai-engineer-ss-6-word-segmentation


In [5]:
# 4) DATA PARSING

@dataclass
class SegmentSample:
    text: str
    end_labels: np.ndarray  # 1 if this char ends a word else 0


def word_to_end_labels(word: str) -> List[int]:
    chars = list(word)
    labels = [0] * len(chars)
    labels[-1] = 1
    return labels


def parse_lst20_split(split_dir: Path) -> List[SegmentSample]:
    """
    Parse one LST20 split and split every sentence into NO-SPACE segments.
    Spaces are represented as '_' in the corpus and are not predicted in the competition.
    """
    samples: List[SegmentSample] = []

    for fp in sorted(split_dir.glob("T*.txt")):
        cur_words: List[str] = []

        def flush_segment():
            nonlocal cur_words, samples
            if not cur_words:
                return
            text_parts = []
            end_parts = []
            for w in cur_words:
                text_parts.extend(list(w))
                end_parts.extend(word_to_end_labels(w))
            samples.append(SegmentSample(
                text="".join(text_parts),
                end_labels=np.asarray(end_parts, dtype=np.int64)
            ))
            cur_words = []

        for line in fp.read_text(encoding="utf-8", errors="ignore").splitlines():
            line = line.rstrip("\n")
            if not line.strip():
                flush_segment()
                continue

            tok = line.split("\t")[0]
            if tok == "_":
                flush_segment()
            else:
                cur_words.append(tok)

        flush_segment()

    return samples


print("Parsing train/eval...")
train_samples = parse_lst20_split(LST20_ROOT_PATH / "train")
valid_samples = parse_lst20_split(LST20_ROOT_PATH / "eval")
print("train segments:", len(train_samples))
print("valid segments:", len(valid_samples))
print("sample:", train_samples[0].text[:50], train_samples[0].end_labels[:20])


Parsing train/eval...
train segments: 465690
valid segments: 45540
sample: สุรยุทธ์ยันปฏิเสธลงนาม [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0]


In [6]:
# 5) VOCAB + CHAR TYPES

def char_type(ch: str) -> int:
    # 0 pad, 1 Thai base, 2 Thai mark, 3 digit, 4 latin, 5 punct/symbol, 6 other
    cp = ord(ch)
    cat = unicodedata.category(ch)
    if 0x0E00 <= cp <= 0x0E7F:
        if cat.startswith("M"):
            return 2
        return 1
    if ch.isdigit():
        return 3
    if ("A" <= ch <= "Z") or ("a" <= ch <= "z"):
        return 4
    if cat.startswith("P") or cat.startswith("S"):
        return 5
    return 6


def build_vocab(samples_a: List[SegmentSample], samples_b: List[SegmentSample], test_text: str):
    charset = set()
    for s in samples_a:
        charset.update(s.text)
    for s in samples_b:
        charset.update(s.text)
    charset.update(test_text.replace(" ", ""))
    chars = sorted(charset)
    stoi = {"<PAD>": 0, "<UNK>": 1}
    for i, ch in enumerate(chars, start=2):
        stoi[ch] = i
    itos = {i: s for s, i in stoi.items()}
    return stoi, itos


test_text_full = (COMP_ROOT_PATH / "ws_test.txt").read_text(encoding="utf-8")
stoi, itos = build_vocab(train_samples, valid_samples, test_text_full)
print("vocab size:", len(stoi))



vocab size: 181


In [7]:
# 6) DATASET / COLLATE

class BoundaryDataset(Dataset):
    def __init__(self, samples: List[SegmentSample], stoi: Dict[str, int]):
        self.samples = samples
        self.stoi = stoi

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        x = [self.stoi.get(ch, 1) for ch in s.text]
        t = [char_type(ch) for ch in s.text]
        y = s.end_labels.astype(np.float32)
        return {
            "x": torch.tensor(x, dtype=torch.long),
            "t": torch.tensor(t, dtype=torch.long),
            "y": torch.tensor(y, dtype=torch.float32),
            "text": s.text,
        }


def collate_fn(batch):
    lengths = torch.tensor([len(b["x"]) for b in batch], dtype=torch.long)
    max_len = int(lengths.max())

    xs = torch.zeros(len(batch), max_len, dtype=torch.long)
    ts = torch.zeros(len(batch), max_len, dtype=torch.long)
    ys = torch.zeros(len(batch), max_len, dtype=torch.float32)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    texts = []

    for i, b in enumerate(batch):
        L = len(b["x"])
        xs[i, :L] = b["x"]
        ts[i, :L] = b["t"]
        ys[i, :L] = b["y"]
        mask[i, :L] = True
        texts.append(b["text"])

    return {
        "x": xs,
        "t": ts,
        "y": ys,
        "mask": mask,
        "lengths": lengths,
        "texts": texts,
    }


train_ds = BoundaryDataset(train_samples, stoi)
valid_ds = BoundaryDataset(valid_samples, stoi)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)



In [8]:
# 7) MODEL

class CharBoundaryBiLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, type_embed_dim: int, hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.char_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.type_emb = nn.Embedding(7, type_embed_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim + type_embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=True,
        )
        self.norm = nn.LayerNorm(hidden_dim * 2)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, t, lengths):
        emb = torch.cat([self.char_emb(x), self.type_emb(t)], dim=-1)
        emb = self.emb_dropout(emb)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        out = self.norm(out)
        logits = self.head(out).squeeze(-1)
        return logits


In [9]:
# 8) DECODING + METRICS

LABEL_B = "B_WORD"
LABEL_I = "I_WORD"
LABEL_E = "E_WORD"


def boundaries_to_labels(boundaries: np.ndarray, single_char_label: str = "B_WORD") -> List[str]:
    n = len(boundaries)
    if n == 0:
        return []
    b = boundaries.astype(np.int64).copy()
    b[-1] = 1  # final char must end a word

    labels = []
    start = 0
    ends = np.where(b == 1)[0]
    for end in ends:
        L = end - start + 1
        if L <= 0:
            continue
        if L == 1:
            labels.append(single_char_label)
        elif L == 2:
            labels.extend([LABEL_B, LABEL_E])
        else:
            labels.append(LABEL_B)
            labels.extend([LABEL_I] * (L - 2))
            labels.append(LABEL_E)
        start = end + 1

    # In case of numerical issues, make sure length matches.
    if len(labels) != n:
        # fallback: greedy repair
        labels = []
        start = 0
        for i in range(n):
            if i == n - 1 or b[i] == 1:
                L = i - start + 1
                if L == 1:
                    labels.append(single_char_label)
                elif L == 2:
                    labels.extend([LABEL_B, LABEL_E])
                else:
                    labels.append(LABEL_B)
                    labels.extend([LABEL_I] * (L - 2))
                    labels.append(LABEL_E)
                start = i + 1
    assert len(labels) == n
    return labels


def probs_to_boundaries(probs: np.ndarray, threshold: float) -> np.ndarray:
    b = (probs >= threshold).astype(np.int64)
    b[-1] = 1
    return b


def evaluate_macro_f1(model, loader, threshold: float, single_char_label: str = "B_WORD"):
    model.eval()
    y_true_all, y_pred_all = [], []
    loss_sum, n_batches = 0.0, 0
    bce = nn.BCEWithLogitsLoss(reduction="none")

    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            t = batch["t"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)

            logits = model(x, t, lengths)
            loss = bce(logits, y)
            loss = (loss * mask.float()).sum() / mask.sum().clamp_min(1)
            loss_sum += float(loss.item())
            n_batches += 1

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            y_np = y.detach().cpu().numpy().astype(np.int64)
            lengths_np = batch["lengths"].numpy()

            for i, L in enumerate(lengths_np):
                true_bound = y_np[i, :L]
                pred_bound = probs_to_boundaries(probs[i, :L], threshold)
                true_labels = boundaries_to_labels(true_bound, single_char_label=single_char_label)
                pred_labels = boundaries_to_labels(pred_bound, single_char_label=single_char_label)
                y_true_all.extend(true_labels)
                y_pred_all.extend(pred_labels)

    score = f1_score(y_true_all, y_pred_all, average="macro", labels=[LABEL_B, LABEL_I, LABEL_E])
    return score, loss_sum / max(n_batches, 1)


def find_best_threshold(model, loader, single_char_label: str = "B_WORD"):
    best_t, best_s = 0.5, -1.0
    all_scores = []
    for t in THRESHOLDS:
        s, _ = evaluate_macro_f1(model, loader, threshold=float(t), single_char_label=single_char_label)
        all_scores.append((float(t), float(s)))
        if s > best_s:
            best_s = s
            best_t = float(t)
    return best_t, best_s, all_scores


In [10]:
# 9) TRAINING

def train_model(train_loader, valid_loader, vocab_size: int, lr: float, epochs: int, patience: int, model_path: str):
    model = CharBoundaryBiLSTM(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        type_embed_dim=TYPE_EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    scaler = GradScaler(enabled=USE_AMP)
    bce = nn.BCEWithLogitsLoss(reduction="none")

    best_score = -1.0
    best_epoch = 0
    best_thr = 0.5
    bad_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            x = batch["x"].to(DEVICE, non_blocking=True)
            t = batch["t"].to(DEVICE, non_blocking=True)
            y = batch["y"].to(DEVICE, non_blocking=True)
            mask = batch["mask"].to(DEVICE, non_blocking=True)
            lengths = batch["lengths"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP):
                logits = model(x, t, lengths)
                loss = bce(logits, y)
                loss = (loss * mask.float()).sum() / mask.sum().clamp_min(1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.item())
            n_batches += 1

        train_loss = running_loss / max(n_batches, 1)
        thr, score, thr_grid = find_best_threshold(model, valid_loader, single_char_label="B_WORD")
        val_score_b, val_loss = evaluate_macro_f1(model, valid_loader, threshold=thr, single_char_label="B_WORD")
        val_score_e, _ = evaluate_macro_f1(model, valid_loader, threshold=thr, single_char_label="E_WORD")
        scheduler.step(val_score_b)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_f1_singleB": val_score_b,
            "val_macro_f1_singleE": val_score_e,
            "threshold": thr,
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"[Epoch {epoch:02d}] train_loss={train_loss:.5f} | "
            f"val_f1(B)={val_score_b:.6f} | val_f1(E)={val_score_e:.6f} | thr={thr:.2f} | lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if val_score_b > best_score:
            best_score = val_score_b
            best_epoch = epoch
            best_thr = thr
            bad_epochs = 0
            torch.save({
                "model_state": model.state_dict(),
                "best_epoch": best_epoch,
                "best_thr": best_thr,
                "best_score": best_score,
                "history": history,
                "stoi": stoi,
            }, model_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping triggered.")
                break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    ckpt = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    return model, ckpt


stage1_model_path = str(Path(OUT_DIR) / "best_stage1_boundary_model.pt")
stage1_model, stage1_ckpt = train_model(
    train_loader=train_loader,
    valid_loader=valid_loader,
    vocab_size=len(stoi),
    lr=LR,
    epochs=EPOCHS,
    patience=PATIENCE,
    model_path=stage1_model_path,
)

print("Stage-1 best epoch:", stage1_ckpt["best_epoch"])
print("Stage-1 best threshold:", stage1_ckpt["best_thr"])
print("Stage-1 best score:", stage1_ckpt["best_score"])

pd.DataFrame(stage1_ckpt["history"]).to_csv(Path(OUT_DIR) / "training_history_stage1.csv", index=False)




/tmp/ipykernel_11437/1816121467.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)
/tmp/ipykernel_11437/1816121467.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


[Epoch 01] train_loss=0.03737 | val_f1(B)=0.985392 | val_f1(E)=0.985535 | thr=0.49 | lr=2.00e-03


/tmp/ipykernel_11437/1816121467.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


[Epoch 02] train_loss=0.01669 | val_f1(B)=0.987612 | val_f1(E)=0.987799 | thr=0.39 | lr=2.00e-03


/tmp/ipykernel_11437/1816121467.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


...

[Epoch 06] train_loss=0.01198 | val_f1(B)=0.988676 | val_f1(E)=0.988846 | thr=0.37 | lr=2.00e-03


/tmp/ipykernel_11437/1816121467.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


[Epoch 07] train_loss=0.01154 | val_f1(B)=0.988898 | val_f1(E)=0.989063 | thr=0.49 | lr=2.00e-03


/tmp/ipykernel_11437/1816121467.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


[Epoch 08] train_loss=0.01135 | val_f1(B)=0.988832 | val_f1(E)=0.988999 | thr=0.51 | lr=2.00e-03
Stage-1 best epoch: 7
Stage-1 best threshold: 0.4900000000000001
Stage-1 best score: 0.9888976080450184


In [11]:
# 10) FINAL RETRAIN ON TRAIN+EVAL (OPTIONAL)

final_model = stage1_model
final_threshold = float(stage1_ckpt["best_thr"])

if FINAL_RETRAIN:
    print("\nRetraining final model on train+eval for", stage1_ckpt["best_epoch"], "epochs ...")
    all_samples = train_samples + valid_samples
    final_ds = BoundaryDataset(all_samples, stoi)
    final_loader = DataLoader(
        final_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )

    final_model = CharBoundaryBiLSTM(
        vocab_size=len(stoi),
        embed_dim=EMBED_DIM,
        type_embed_dim=TYPE_EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(final_model.parameters(), lr=FINAL_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(enabled=USE_AMP)
    bce = nn.BCEWithLogitsLoss(reduction="none")

    final_model.train()
    for epoch in range(1, int(stage1_ckpt["best_epoch"]) + 1):
        running_loss = 0.0
        n_batches = 0
        for batch in final_loader:
            x = batch["x"].to(DEVICE, non_blocking=True)
            t = batch["t"].to(DEVICE, non_blocking=True)
            y = batch["y"].to(DEVICE, non_blocking=True)
            mask = batch["mask"].to(DEVICE, non_blocking=True)
            lengths = batch["lengths"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP):
                logits = final_model(x, t, lengths)
                loss = bce(logits, y)
                loss = (loss * mask.float()).sum() / mask.sum().clamp_min(1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(final_model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.item())
            n_batches += 1

        print(f"[Final Epoch {epoch:02d}] train_loss={running_loss / max(n_batches,1):.5f}")

    torch.save({
        "model_state": final_model.state_dict(),
        "threshold": final_threshold,
        "stoi": stoi,
    }, Path(OUT_DIR) / "final_boundary_model.pt")





Retraining final model on train+eval for 7 epochs ...


/tmp/ipykernel_11437/3103012296.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)
/tmp/ipykernel_11437/3103012296.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


[Final Epoch 01] train_loss=0.03585
[Final Epoch 02] train_loss=0.01607
[Final Epoch 03] train_loss=0.01358
[Final Epoch 04] train_loss=0.01227
[Final Epoch 05] train_loss=0.01137
[Final Epoch 06] train_loss=0.01080
[Final Epoch 07] train_loss=0.01038


In [12]:
# 11) TEST PREP

sample_sub = pd.read_csv(COMP_ROOT_PATH / "ws_sample_submission.csv")
assert len(test_text_full) == 37248, f"Unexpected ws_test length: {len(test_text_full)}"
assert len(sample_sub) == 35182, f"Unexpected sample submission rows: {len(sample_sub)}"


def split_test_into_nonspace_segments(text: str):
    segments = []
    cur_chars = []
    cur_ids = []
    for idx, ch in enumerate(text, start=1):
        if ch == " ":
            if cur_chars:
                segments.append(("".join(cur_chars), cur_ids.copy()))
                cur_chars = []
                cur_ids = []
        else:
            cur_chars.append(ch)
            cur_ids.append(idx)
    if cur_chars:
        segments.append(("".join(cur_chars), cur_ids.copy()))
    return segments


test_segments = split_test_into_nonspace_segments(test_text_full)
print("test non-space segments:", len(test_segments))
print("first segment:", test_segments[0][0][:60], test_segments[0][1][:10])



test non-space segments: 2067
first segment: ที่ยังสถานการณ์ยังไม่คลี่คลายอาจส่งผลกระทบการค้าชายแดนไทย [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [13]:
# 12) INFERENCE

class PredictDataset(Dataset):
    def __init__(self, seg_texts: List[str], stoi: Dict[str, int]):
        self.seg_texts = seg_texts
        self.stoi = stoi

    def __len__(self):
        return len(self.seg_texts)

    def __getitem__(self, idx):
        text = self.seg_texts[idx]
        x = [self.stoi.get(ch, 1) for ch in text]
        t = [char_type(ch) for ch in text]
        return {
            "x": torch.tensor(x, dtype=torch.long),
            "t": torch.tensor(t, dtype=torch.long),
            "text": text,
        }


def pred_collate_fn(batch):
    lengths = torch.tensor([len(b["x"]) for b in batch], dtype=torch.long)
    max_len = int(lengths.max())
    xs = torch.zeros(len(batch), max_len, dtype=torch.long)
    ts = torch.zeros(len(batch), max_len, dtype=torch.long)
    texts = []
    for i, b in enumerate(batch):
        L = len(b["x"])
        xs[i, :L] = b["x"]
        ts[i, :L] = b["t"]
        texts.append(b["text"])
    return {"x": xs, "t": ts, "lengths": lengths, "texts": texts}


def predict_boundaries_for_segments(model, segments: List[Tuple[str, List[int]]], threshold: float):
    seg_texts = [s for s, _ in segments]
    ds = PredictDataset(seg_texts, stoi)
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=pred_collate_fn,
    )

    all_boundaries = []
    model.eval()
    with torch.no_grad():
        for batch in dl:
            x = batch["x"].to(DEVICE, non_blocking=True)
            t = batch["t"].to(DEVICE, non_blocking=True)
            lengths = batch["lengths"].to(DEVICE, non_blocking=True)
            logits = model(x, t, lengths)
            probs = torch.sigmoid(logits).cpu().numpy()
            lengths_np = batch["lengths"].numpy()
            for i, L in enumerate(lengths_np):
                bound = probs_to_boundaries(probs[i, :L], threshold)
                all_boundaries.append(bound)
    return all_boundaries


pred_boundaries = predict_boundaries_for_segments(final_model, test_segments, threshold=final_threshold)
assert len(pred_boundaries) == len(test_segments)



In [14]:
# 13) BUILD SUBMISSION FILES

def make_submission_df(sample_sub: pd.DataFrame, segments: List[Tuple[str, List[int]]], boundaries_list: List[np.ndarray], single_char_label: str):
    id_to_label = {}
    for (seg_text, seg_ids), seg_bound in zip(segments, boundaries_list):
        labels = boundaries_to_labels(seg_bound, single_char_label=single_char_label)
        assert len(labels) == len(seg_text) == len(seg_ids)
        for idx, lab in zip(seg_ids, labels):
            id_to_label[idx] = lab

    sub = sample_sub.copy()
    sub["Predicted"] = sub["Id"].map(id_to_label)
    assert sub["Predicted"].isna().sum() == 0, "Some ids were not assigned predictions"
    assert len(sub) == 35182
    return sub


sub_B = make_submission_df(sample_sub, test_segments, pred_boundaries, single_char_label="B_WORD")
sub_E = make_submission_df(sample_sub, test_segments, pred_boundaries, single_char_label="E_WORD")

sub_B.to_csv(Path(OUT_DIR) / "submission_singleB.csv", index=False)
sub_E.to_csv(Path(OUT_DIR) / "submission_singleE.csv", index=False)

# default submission.csv
if DEFAULT_SINGLE_CHAR_LABEL == "B_WORD":
    sub_B.to_csv(Path(OUT_DIR) / "submission.csv", index=False)
else:
    sub_E.to_csv(Path(OUT_DIR) / "submission.csv", index=False)

print("Saved:")
print(Path(OUT_DIR) / "submission.csv")
print(Path(OUT_DIR) / "submission_singleB.csv")
print(Path(OUT_DIR) / "submission_singleE.csv")
print(sub_B.head(10))


Saved:
/content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/submission/submission.csv
/content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/submission/submission_singleB.csv
/content/drive/MyDrive/Learn/SuperAI_Season6/WordSegmentation/submission/submission_singleE.csv
   Id Predicted
0   1    B_WORD
1   2    I_WORD
2   3    E_WORD
3   4    B_WORD
4   5    I_WORD
5   6    E_WORD
6   7    B_WORD
7   8    I_WORD
8   9    I_WORD
9  10    I_WORD


In [15]:
# 14) QUICK SANITY CHECKS

assert set(sub_B["Predicted"].unique()).issubset({"B_WORD", "I_WORD", "E_WORD"})
assert set(sub_E["Predicted"].unique()).issubset({"B_WORD", "I_WORD", "E_WORD"})
assert len(sub_B) == 35182 and len(sub_E) == 35182
assert sub_B["Id"].equals(sample_sub["Id"]) and sub_E["Id"].equals(sample_sub["Id"])
print("All checks passed.")


All checks passed.
